In [8]:
import os
import numpy as np
import librosa
import tensorflow as tf
import xml.etree.ElementTree as ET
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import NMF
from sklearn.metrics import classification_report
import re


# Function to parse annotations from RML files
def parse_annotations(rml_file_path):
    annotations = []
    try:
        tree = ET.parse(rml_file_path)
        root = tree.getroot()
        for event in root.findall(".//ns0:Event", namespaces={"ns0": "http://www.respironics.com/PatientStudy.xsd"}):
            family = event.attrib.get("Family", "").strip()
            type_ = event.attrib.get("Type", "").strip()
            start = event.attrib.get("Start", "0").strip()
            duration = event.attrib.get("Duration", "0").strip()
            try:
                start = float(start)
                duration = float(duration)
            except ValueError:
                print(f"Invalid Start or Duration in annotation: Start={start}, Duration={duration}")
                continue
            if family == "Respiratory":
                annotations.append({"family": family, "type": type_, "start": start, "duration": duration})
    except Exception as e:
        print(f"Error parsing {rml_file_path}: {e}")
    return annotations


# Function to extract Mel spectrograms
def extract_mel_spectrogram(wav_file_path, sr=22050, n_mels=128, hop_length=512):
    try:
        audio, _ = librosa.load(wav_file_path, sr=sr)
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels, hop_length=hop_length)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Shift values to be non-negative
        mel_spec_db -= mel_spec_db.min()
        return mel_spec_db
    except Exception as e:
        print(f"Error extracting Mel spectrogram: {e}")
        return None


# Function to apply NMF decomposition
def apply_nmf(mel_spectrogram, n_components=10):
    try:
        nmf = NMF(n_components=n_components, random_state=42)
        W = nmf.fit_transform(mel_spectrogram.T)
        H = nmf.components_
        return W, H
    except Exception as e:
        print(f"Error applying NMF: {e}")
        return None, None


# Focal loss function
def focal_loss(gamma=2., alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=tf.shape(y_pred)[-1])
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)  # Avoid log(0)

        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        fl = -alpha_t * tf.pow((1 - p_t), gamma) * tf.math.log(p_t)
        return tf.reduce_mean(fl)
    return focal_loss_fixed


# Function to build LSTM model
def build_lstm_model(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.LSTM(64),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model


# Main function to process data and train LSTM model
def process_and_train_lstm(wav_folder_path, rml_folder_path, sr=22050, n_components=10, n_splits=5, hop_length=512):
    all_features = []
    all_labels = []

    # Match .wav and .rml files by identifier
    wav_files = {re.match(r"(.*)_mic_cleaned\.wav", f).group(1): f
                 for f in os.listdir(wav_folder_path) if f.endswith("_mic_cleaned.wav")}
    rml_files = {re.match(r"filtered_(.*)\.rml", f).group(1): f
                 for f in os.listdir(rml_folder_path) if f.endswith(".rml")}

    common_ids = set(wav_files.keys()).intersection(rml_files.keys())
    for file_id in common_ids:
        wav_file_path = os.path.join(wav_folder_path, wav_files[file_id])
        rml_file_path = os.path.join(rml_folder_path, rml_files[file_id])

        print(f"Processing: ID={file_id}, WAV={wav_files[file_id]}, RML={rml_files[file_id]}")
        annotations = parse_annotations(rml_file_path)
        mel_spectrogram = extract_mel_spectrogram(wav_file_path, sr, hop_length=hop_length)
        if mel_spectrogram is None:
            continue

        W, _ = apply_nmf(mel_spectrogram, n_components=n_components)
        if W is None or len(W) == 0:
            continue

        for annotation in annotations:
            start_time = annotation['start']
            duration = annotation['duration']
            end_time = start_time + duration

            start_frame = int(start_time * sr / hop_length)
            end_frame = int(end_time * sr / hop_length)

            if start_frame >= W.shape[0] or end_frame > W.shape[0]:
                print(f"Skipping annotation outside spectrogram bounds: Start={start_frame}, End={end_frame}")
                continue

            # Aggregate NMF components over the annotation's time range
            feature_vector = W[start_frame:end_frame].mean(axis=0)
            all_features.append(feature_vector)
            all_labels.append(annotation["type"])

    # Validate alignment
    if len(all_features) != len(all_labels):
        raise ValueError(f"Final mismatch: Features={len(all_features)}, Labels={len(all_labels)}")

    all_features = np.array(all_features)
    all_labels = np.array(all_labels)

    # Reshape features for LSTM input
    all_features = all_features[..., np.newaxis]

    label_encoder = LabelEncoder()
    all_labels_encoded = label_encoder.fit_transform(all_labels)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for fold, (train_idx, test_idx) in enumerate(skf.split(all_features, all_labels_encoded)):
        print(f"\nFold {fold + 1}/{n_splits}")

        X_train, X_test = all_features[train_idx], all_features[test_idx]
        y_train, y_test = all_labels_encoded[train_idx], all_labels_encoded[test_idx]

        # Debug input shapes
        print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
        print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

        model = build_lstm_model((X_train.shape[1], X_train.shape[2]), len(label_encoder.classes_))
        model.compile(optimizer='adam', loss=focal_loss(), metrics=['accuracy'])

        model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=32)

        loss, accuracy = model.evaluate(X_test, y_test)
        print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

        y_pred = model.predict(X_test)
        y_pred_classes = np.argmax(y_pred, axis=1)
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))


# Specify folder paths and execute
wav_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_wav/Mic"
rml_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_label"

process_and_train_lstm(wav_folder_path, rml_folder_path)


Processing: ID=00000995-100507, WAV=00000995-100507_mic_cleaned.wav, RML=filtered_00000995-100507.rml
Processing: ID=00001026-100507, WAV=00001026-100507_mic_cleaned.wav, RML=filtered_00001026-100507.rml
Processing: ID=00001112-100507, WAV=00001112-100507_mic_cleaned.wav, RML=filtered_00001112-100507.rml
Processing: ID=00001290-100507, WAV=00001290-100507_mic_cleaned.wav, RML=filtered_00001290-100507.rml
Processing: ID=00001449-100507, WAV=00001449-100507_mic_cleaned.wav, RML=filtered_00001449-100507.rml
Processing: ID=00001172-100507, WAV=00001172-100507_mic_cleaned.wav, RML=filtered_00001172-100507.rml
Processing: ID=00001367-100507, WAV=00001367-100507_mic_cleaned.wav, RML=filtered_00001367-100507.rml
Processing: ID=00001239-100507, WAV=00001239-100507_mic_cleaned.wav, RML=filtered_00001239-100507.rml
Processing: ID=00001308-100507, WAV=00001308-100507_mic_cleaned.wav, RML=filtered_00001308-100507.rml
Processing: ID=00001318-100507, WAV=00001318-100507_mic_cleaned.wav, RML=filtered_

/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


1091/1091 ━━━━━━━━━━━━━━━━━━━━ 28s 24ms/step - accuracy: 0.6010 - loss: 0.0380 - val_accuracy: 0.5990 - val_loss: 0.0364
Epoch 2/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5973 - loss: 0.0365 - val_accuracy: 0.6007 - val_loss: 0.0357
Epoch 3/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5970 - loss: 0.0360 - val_accuracy: 0.5997 - val_loss: 0.0354
Epoch 4/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5989 - loss: 0.0355 - val_accuracy: 0.6006 - val_loss: 0.0352
Epoch 5/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5988 - loss: 0.0353 - val_accuracy: 0.6014 - val_loss: 0.0350
Epoch 6/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5993 - loss: 0.0352 - val_accuracy: 0.6014 - val_loss: 0.0347
Epoch 7/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.6026 - loss: 0.0351 - val_accuracy: 0.6038 - val_loss: 0.0347
Epoch 8/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.6054 - loss: 0.03

/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


1091/1091 ━━━━━━━━━━━━━━━━━━━━ 28s 24ms/step - accuracy: 0.5935 - loss: 0.0382 - val_accuracy: 0.5990 - val_loss: 0.0366
Epoch 2/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.6037 - loss: 0.0359 - val_accuracy: 0.5987 - val_loss: 0.0357
Epoch 3/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5977 - loss: 0.0358 - val_accuracy: 0.5995 - val_loss: 0.0357
Epoch 4/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5969 - loss: 0.0355 - val_accuracy: 0.5992 - val_loss: 0.0352
Epoch 5/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.6009 - loss: 0.0352 - val_accuracy: 0.5993 - val_loss: 0.0352
Epoch 6/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.6014 - loss: 0.0351 - val_accuracy: 0.6028 - val_loss: 0.0348
Epoch 7/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5974 - loss: 0.0350 - val_accuracy: 0.6021 - val_loss: 0.0346
Epoch 8/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.6056 - loss: 0.03

/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


1091/1091 ━━━━━━━━━━━━━━━━━━━━ 31s 27ms/step - accuracy: 0.5987 - loss: 0.0384 - val_accuracy: 0.6013 - val_loss: 0.0361
Epoch 2/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 29s 26ms/step - accuracy: 0.5989 - loss: 0.0361 - val_accuracy: 0.5952 - val_loss: 0.0359
Epoch 3/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.5972 - loss: 0.0357 - val_accuracy: 0.5916 - val_loss: 0.0360
Epoch 4/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.5940 - loss: 0.0357 - val_accuracy: 0.5991 - val_loss: 0.0353
Epoch 5/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.5967 - loss: 0.0352 - val_accuracy: 0.5999 - val_loss: 0.0351
Epoch 6/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.6020 - loss: 0.0350 - val_accuracy: 0.6031 - val_loss: 0.0349
Epoch 7/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.6023 - loss: 0.0348 - val_accuracy: 0.6026 - val_loss: 0.0348
Epoch 8/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.6015 - loss: 0.03

/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


1091/1091 ━━━━━━━━━━━━━━━━━━━━ 28s 25ms/step - accuracy: 0.5951 - loss: 0.0384 - val_accuracy: 0.5992 - val_loss: 0.0364
Epoch 2/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 27s 24ms/step - accuracy: 0.5978 - loss: 0.0364 - val_accuracy: 0.6000 - val_loss: 0.0358
Epoch 3/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.6011 - loss: 0.0357 - val_accuracy: 0.6001 - val_loss: 0.0354
Epoch 4/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5987 - loss: 0.0355 - val_accuracy: 0.5865 - val_loss: 0.0355
Epoch 5/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 27s 24ms/step - accuracy: 0.6036 - loss: 0.0351 - val_accuracy: 0.5968 - val_loss: 0.0352
Epoch 6/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 27s 24ms/step - accuracy: 0.6016 - loss: 0.0351 - val_accuracy: 0.6000 - val_loss: 0.0350
Epoch 7/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 27s 24ms/step - accuracy: 0.6018 - loss: 0.0350 - val_accuracy: 0.6009 - val_loss: 0.0347
Epoch 8/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 27s 24ms/step - accuracy: 0.6051 - loss: 0.03

/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
